# UMUD R17C — ocena i odizolowane próby PA/FL
Uruchom **Run all w Colab GPU**, po R17B. Wykorzystuje oryginalny segmenter apo R12B i nowy segmenter pęczków.
Najpierw zapisuje test segmentacji na zablokowanym foldzie 0, następnie diagnostykę OSF-35 i predykcje 309 obrazów.
**Domyślne zgłoszenie zmienia tylko PA.** FL i MT pozostają dokładnie takie jak w bazie 0,49212.

To pakiet eksperymentalny. Hybrydowe zgłoszenie zawiera wcześniejsze predykcje i kalibrację MT;
nie jest deklarowane jako gotowe zgłoszenie finałowe na nowy zbiór. Surowe pomiary z obrazów są zapisane oddzielnie.
Funkcja `infer_manifest` działa dla nowych obrazów z jawną skalą px/mm; nie zawiera wyszukiwania predykcji po ID.

In [7]:
from pathlib import Path
import sys, os, json, time, hashlib, importlib.util, subprocess
RUN_NAME = "R17_v1_0_0"
PROJECT = Path("/content/drive/MyDrive/UMUD_Challenge")
if not Path("/content/drive/MyDrive").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("BLOCKED: Drive nie jest zamontowany.")
RUN = PROJECT / "outputs" / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
missing=[]
for mod,pkg in [("numpy","numpy"),("pandas","pandas"),("cv2","opencv-python-headless"),
                ("scipy","scipy"),("skimage","scikit-image"),("matplotlib","matplotlib")]:
    if importlib.util.find_spec(mod) is None: missing.append(pkg)
if missing: subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
print("Wyniki:", RUN)

Wyniki: /content/drive/MyDrive/UMUD_Challenge/outputs/R17_v1_0_0


In [8]:
# Pakiet zawiera jawne źródła Python, zamrożone grupy i małe dane odniesienia.
# Jest identyczny z katalogami src/ i assets/ w dostarczonym ZIP.
import base64, zipfile, io
package_bytes=base64.b64decode('...')
assert hashlib.sha256(package_bytes).hexdigest()=='2343c540b3222cdeb320a807a88b2706ebbc348597226331fcc581f46f1e7d08'
with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
    for name in archive.namelist():
        path=RUN/name
        assert path.resolve().is_relative_to(RUN.resolve())
        path.parent.mkdir(parents=True,exist_ok=True)
        path.write_bytes(archive.read(name))
if str(RUN/"code") not in sys.path:sys.path.insert(0,str(RUN/"code"))
for name in ["umud_infer","umud_audit","umud_model","umud_core"]:
    sys.modules.pop(name,None)
from umud_core import *
print("Test geometrii:",geometry_selftest())

Test geometrii: {'pa_deg': 30.004681504849913, 'fl_mm': 31.195585247451312, 'mt_mm': 15.6, 'mt_center_mm': 16.0, 'band_correction_mm': 0.4, 'n_fasc': 1, 'apo_pairs': 1, 'extrapolation_ratio': 0.6773636801416603, 'status': 'OK'}


In [9]:
import torch
if importlib.util.find_spec("timm") is None:subprocess.check_call([sys.executable,"-m","pip","install","-q","timm"])
from umud_model import *
from umud_audit import stage_cache
from umud_infer import *
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_KIND = "refit"       # zmień na "refit" dopiero po takim treningu B
EXPORT_TARGETS = "pa_only"        # pa_only / fl_only / pafl; jeden CSV zgłoszeniowy na przebieg
APO_CHECKPOINT=Path("/content/drive/MyDrive/UMUD_R12B/best_dinov2_frozen.pt")
if not APO_CHECKPOINT.is_file():raise FileNotFoundError(APO_CHECKPOINT)
FASC_CHECKPOINT=RUN/CHECKPOINT_KIND/"best.pt"
selection=json.loads((RUN/CHECKPOINT_KIND/"selection.json").read_text())
assert sha256(FASC_CHECKPOINT)==selection["checkpoint_sha256"]
THRESHOLD=float(selection["threshold"])
print("Model:",FASC_CHECKPOINT,"próg:",THRESHOLD,"eksport:",EXPORT_TARGETS)

Model: /content/drive/MyDrive/UMUD_Challenge/outputs/R17_v1_0_0/refit/best.pt próg: 0.65 eksport: pa_only


## 1. Zamrożony test segmentacji
Oceniamy wyłącznie nową gałąź fasc. Stary model apo mógł widzieć część odpowiadających obrazów,
dlatego nie nazywamy połączonego PA/FL walidacją OOF.
Fold 0 to jedna grupa wizualna, 370 unikalnych obrazów; wynik trzeba tak interpretować.

In [10]:
if CHECKPOINT_KIND=="develop":
    samples=pd.read_csv(RUN/"samples.csv");outer=samples[samples.fold==0].copy()
    audit=json.loads((RUN/"AUDIT.json").read_text())
    local=Path("/content/umud_r17_cache")/audit["cache_identity"][:16]
    stage_cache(outer,RUN/"cache",local)
    model,_=load_model(FASC_CHECKPOINT,DEVICE)
    metrics=evaluate(model,outer,local,DEVICE,thresholds=(THRESHOLD,))
    metrics.to_csv(RUN/"outer_segmentation.csv",index=False)
    summary=summarize_evaluation(metrics)
    atomic_json(RUN/"outer_report.json",dict(checkpoint_sha256=sha256(FASC_CHECKPOINT),summary=summary,
        interpretation="single visual group; fasc-only; no scalar competition labels"))
    print(summary);del model
    if DEVICE=="cuda":torch.cuda.empty_cache()
else:
    print("Refit widział wszystkie dane treningowe; nie obliczamy pozornego holdoutu.")

Refit widział wszystkie dane treningowe; nie obliczamy pozornego holdoutu.


## 2. OSF-35 — wyłącznie diagnostyka transferu
Ten zbiór był wielokrotnie używany w projekcie. Raportuje MAE na poprawnych odczytach oraz pokrycie;
nie ekstrapoluje wyniku leaderboardu i nie dobiera parametrów.

In [11]:
ext=pd.read_csv(RUN/"assets/external_truth.csv")
ext_manifest=pd.DataFrame(dict(image_id=ext.image_id,path=ext.path,px_per_mm=ext.px_per_cm/10))
if all(Path(p).is_file() for p in ext_manifest.path):
    ext_pred=infer_manifest(ext_manifest,FASC_CHECKPOINT,APO_CHECKPOINT,RUN/"inference"/CHECKPOINT_KIND/"external",THRESHOLD,DEVICE)
    ext_report=report_external(ext,ext_pred)
    atomic_json(RUN/"inference"/CHECKPOINT_KIND/"external_report.json",ext_report)
    print(json.dumps(ext_report,indent=2))
else:
    print("OSF pominięty: niepełne ścieżki obrazów. Nie blokuje inferencji konkursowej.")

inference 20 / 35
{
  "pa_deg": {
    "n": 34,
    "total": 35,
    "coverage": 0.9714285714285714,
    "conditional_MAE": 1.5139237667116812
  },
  "fl_mm": {
    "n": 34,
    "total": 35,
    "coverage": 0.9714285714285714,
    "conditional_MAE": 7.729879122888953
  },
  "mt_mm": {
    "n": 34,
    "total": 35,
    "coverage": 0.9714285714285714,
    "conditional_MAE": 0.4698797399987626
  },
  "status": "DEVELOPMENT_DIAGNOSTIC_ONLY",
  "warning": "Conditional MAE omits failures. Compare methods only at equal coverage; this is not the competition score."
}


## 3. Obrazy konkursowe → mapy → geometria
PA: kąt pęczka względem dolnego rozcięgna. FL: długość między przecięciami prostych z wewnętrznymi krawędziami.
MT surowe: średnia odległość pionowa krawędź–krawędź; raportujemy też środek–środek i różnicę.
Ta różnica jest mierzona na danym obrazie, nie wymuszona na 1,3629 mm.
W zgłoszeniu eksperymentalnym MT pozostaje jednak z kotwicy, żeby odizolować zmianę PA/FL.

In [12]:
registry=pd.read_csv(RUN/"assets/scale_registry.csv")
anchor=pd.read_csv(RUN/"assets/anchor_049212.csv",dtype=str)
assert len(anchor)==309 and anchor.image_id.is_unique
assert set(anchor.image_id)==set(registry.image_id)
registry=registry.set_index("image_id").loc[anchor.image_id].reset_index()
manifest=pd.DataFrame(dict(image_id=registry.image_id,
    path=[str(PROJECT/"data/competition_test_v1"/name) for name in registry.image_id],
    px_per_mm=registry.SCALE_PX_PER_CM_FINAL.astype(float)/10))
assert np.isfinite(manifest.px_per_mm).all() and (manifest.px_per_mm>0).all()
measurements=infer_manifest(manifest,FASC_CHECKPOINT,APO_CHECKPOINT,RUN/"inference"/CHECKPOINT_KIND/"test",THRESHOLD,DEVICE)
coverage=measurements.groupby(measurements.image_id.str.rsplit(".",n=1).str[-1])[TARGETS].count()
display(coverage)
candidate,changed=compose_candidate(anchor,measurements,EXPORT_TARGETS)
candidate_dir=RUN/"candidates"/CHECKPOINT_KIND/EXPORT_TARGETS
candidate_dir.mkdir(parents=True,exist_ok=True)
output=candidate_dir/"submission.csv"
candidate.to_csv(output,index=False)
roundtrip=pd.read_csv(output,dtype=str)
assert roundtrip.mt_mm.equals(anchor.mt_mm)
if EXPORT_TARGETS=="pa_only":assert roundtrip.fl_mm.equals(anchor.fl_mm)
if EXPORT_TARGETS=="fl_only":assert roundtrip.pa_deg.equals(anchor.pa_deg)
atomic_json(candidate_dir/"experiment.json",dict(mode=EXPORT_TARGETS,changed=changed,rows=len(candidate),
    checkpoint_sha256=sha256(FASC_CHECKPOINT),anchor_sha256=sha256(RUN/"assets/anchor_049212.csv"),csv_sha256=sha256(output),
    public_score=None,final_ready=False,reason="Anchored ablation preserves earlier PA/FL fallbacks and calibrated MT."))
print("ZMIENIONE WIERSZE:",changed)
from IPython.display import FileLink,display
display(FileLink(str(output)))
print("Kolejność prób: pa_only, następnie fl_only; pafl dopiero po ocenie obu. Braki odczytu pozostają z kotwicy.")

inference 20 / 309
inference 40 / 309
inference 60 / 309
inference 80 / 309
inference 100 / 309
inference 120 / 309
inference 140 / 309
inference 160 / 309
inference 180 / 309
inference 200 / 309
inference 220 / 309
inference 240 / 309
inference 260 / 309
inference 280 / 309
inference 300 / 309


,pa_deg,fl_mm,mt_mm
image_id,,,
png,54,54,54
tif,249,249,250


ZMIENIONE WIERSZE: {'pa_deg': 303}


/content/drive/MyDrive/UMUD_Challenge/outputs/R17_v1_0_0/candidates/refit/pa_only/submission.csv

Kolejność prób: pa_only, następnie fl_only; pafl dopiero po ocenie obu. Braki odczytu pozostają z kotwicy.


### Co odesłać do analizy
`AUDIT.json`, `develop/history.csv`, `develop/selection.json`, `outer_report.json`,
`inference/develop/external_report.json`, `inference/develop/test/measurements.csv`,
`candidates/develop/pa_only/experiment.json` oraz wynik publiczny tej próby.
R17 nie przesyła nic automatycznie na Kaggle.